# 📘 Tutorial 4: Cleaning Numeric Data and Missing Values

> This notebook is provided in a clean, non-executed state for readability and reproducibility.

Plotting code should not have to guess whether `"bad"`, missing cells, or impossible values belong on an axis. This tutorial shows how to convert numeric-looking strings, identify missing values, and keep raw and cleaned data separate.

---

## 🎯 Learning objectives

By the end of this notebook, you should be able to:

- detect missing values,
- convert string columns to numeric columns,
- use `dropna` and `fillna` cautiously,
- detect out-of-range values,
- keep a clear distinction between raw and cleaned data.


## 🔧 Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


## 1. Read the messy CSV


In [ ]:
project_root = Path.cwd()
if not (project_root / "data" / "part1").exists():
    project_root = project_root.parent

data_dir = project_root / "data" / "part1"
messy_path = data_dir / "messy_numeric_measurements.csv"

raw = pd.read_csv(messy_path)
raw


In [ ]:
raw.dtypes


The numeric-looking columns have been read as objects because some entries contain missing markers or text.


## 2. Make an explicit cleaned copy


In [ ]:
clean = raw.copy()
clean.head()


Keeping `raw` unchanged lets us inspect the original file again if a cleaning decision needs to be reviewed.


## 3. Convert numeric-looking strings


In [ ]:
numeric_columns = ["time_min", "temperature_C", "response"]

for column in numeric_columns:
    clean[column] = pd.to_numeric(clean[column], errors="coerce")

clean


In [ ]:
clean.dtypes


`errors="coerce"` turns invalid values into missing values. It does not silently invent a number.


## 4. Inspect missing values


In [ ]:
clean.isna().sum()


In [ ]:
clean.loc[clean.isna().any(axis=1)]


Missing values are not always errors. A missing temperature may be less serious than a missing response, depending on the analysis.


## 5. Detect out-of-range values


In [ ]:
response_out_of_range = (clean["response"] < 0) | (clean["response"] > 2)

clean.loc[response_out_of_range, ["sample_id", "time_min", "response"]]


In [ ]:
clean["review_note"] = ""
clean.loc[response_out_of_range, "review_note"] = "response outside expected range"
clean.loc[response_out_of_range, "response"] = np.nan

clean


The rule here is domain-specific. In this toy example, responses below 0 or above 2 are treated as invalid for plotting.


## 6. Drop missing values when the plot requires them


In [ ]:
plot_ready_rows = clean.dropna(subset=["time_min", "response"]).copy()
plot_ready_rows


Dropping rows is a decision. It should be targeted to the columns needed for the next step, not applied blindly to the whole table.


## 7. Fill missing values cautiously


In [ ]:
temperature_median = clean["temperature_C"].median()
clean["temperature_C_filled"] = clean["temperature_C"].fillna(temperature_median)

clean[["sample_id", "time_min", "temperature_C", "temperature_C_filled"]]


Filling can be useful for a supporting variable, but it can also hide missingness. It is often better to keep the original column and add a clearly named filled column.


## 8. Checkpoint: choose a cleaning policy

For this dataset, a reasonable policy is:

- convert numeric-looking columns with `pd.to_numeric`,
- mark invalid responses as missing,
- drop rows only when `time_min` or `response` is missing,
- fill missing temperature only in a separate helper column.


In [ ]:
final_clean = plot_ready_rows[
    ["sample_id", "time_min", "temperature_C", "response", "review_note"]
].reset_index(drop=True)

final_clean


## 🧭 Key takeaways

- Numeric-looking text is not the same as numeric data.
- `pd.to_numeric(..., errors="coerce")` makes invalid values visible as missing values.
- `dropna` and `fillna` are decisions, not automatic fixes.
- Out-of-range rules should be explicit and reviewable.
- Keep raw data unchanged and create cleaned copies in code.
